# FQI Gridworld Checks

This notebook is the exploratory Fitted Q Iteration analysis for the simple gridworld baselines. It builds small tabular MDPs, loads offline datasets, trains `FQISolver`, and inspects the greedy policy learned from the fitted Q-values.

What to look for:
- how tabular state-action features remove representation error in small grids,
- why terminal transitions must be represented in the offline dataset,
- how FQI policy quality changes when dataset coverage changes.


In [18]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import random
import torch
import pandas as pd
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

# Shared setup: keep imports live, fix RNG seeds, and find the project root.
# This lets the notebook run from any working directory inside the thesis repo.
def find_root(current_path):
    current_path = Path(current_path).resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / "src" / "rl_methods").exists() and (parent / "data").exists():
            return parent
    return current_path

PROJECT_ROOT = find_root(Path.cwd())
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
ASSETS_DIR = PROJECT_ROOT / "experiments" / "shared" / "assets"
print(f"Project root found at: {PROJECT_ROOT}")
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
ASSETS_DIR = PROJECT_ROOT / "experiments" / "shared" / "assets"

# Make repo modules importable from notebook kernels launched in subfolders.
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Small default dataset used by the first FQI sanity checks.
DATASET_PATH = DATASETS_DIR / "test_fqi.csv"
print(f"Loading dataset from: {DATASET_PATH}")

from rl_methods import PolicySolver, EnvDataCollector
from rl_methods.fogas import (
    FOGASSolverVectorized,
    FOGASOracleSolverVectorized,
    FOGASHyperOptimizer,
    FOGASEvaluator,
)
from rl_methods.dataset_collection import DatasetAnalyzer
from rl_methods.fqi.fqi_solver import FQISolver
from rl_methods.fqi.fqi_evaluator import FQIEvaluator

seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root found at: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code
Loading dataset from: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/test_fqi.csv
Using device: cpu


## 3 grid

In [3]:
states = torch.arange(9)
actions = torch.arange(4)
N = len(states)  # number of states
A = len(actions) # number of actions
gamma = 0.9

x_0 = 0 # fixed initial state

goal_grid = 8   # absorbing terminal state

def phi(x, a):
    # Convert to scalar if tensor
    if isinstance(x, torch.Tensor): x = x.item()
    if isinstance(a, torch.Tensor): a = a.item()
    
    vec = torch.zeros(N * A, dtype=torch.float64)
    vec[x * A + a] = 1.0
    return vec

omega = torch.zeros(N * A, dtype=torch.float64)
omega[8 * A : 8 * A + 4] = 1.0

# Helper to convert index <-> (row, col)
def to_rc(s): 
    if isinstance(s, torch.Tensor): s = s.item()
    return divmod(s, 3)

def to_s(r, c): 
    return r*3 + c

def next_state(s, a):
    if isinstance(s, torch.Tensor): s = s.item()
    if isinstance(a, torch.Tensor): a = a.item()

    if s == goal_grid:
        return goal_grid  # absorbing

    r, c = to_rc(s)

    if a == 0:  # Up
        r = max(0, r-1)
    elif a == 1:  # Down
        r = min(2, r+1)
    elif a == 2:  # Left
        c = max(0, c-1)
    elif a == 3:  # Right
        c = min(2, c+1)

    return to_s(r, c)

def psi(xp):
    if isinstance(xp, torch.Tensor): xp = xp.item()
    
    v = torch.zeros(N * A, dtype=torch.float64)
    # Iterating over tensors (states/actions) yields 0-d tensors, so we use .item()
    for x in states:
        for a in actions:
            if next_state(x, a) == xp:
                idx = x.item() * A + a.item()
                v[idx] = 1.0
    return v

# Initialize the solver
mdp = PolicySolver(states=states, actions=actions, phi=phi, omega=omega, gamma=gamma, x0=x_0, psi=psi)

In [6]:
# Initialize FQI Solver
solver_fqi = FQISolver(
    mdp=mdp,
    csv_path=str(str(DATASETS_DIR / "grid3_problem.csv")),
    device=device,
    seed=seed,
    ridge=1e-6 
)
evaluator_fqi = FQIEvaluator(solver_fqi)
pi_fqi = solver_fqi.run(
    K=1000, 
    tau=0.1, 
    verbose=True
)
print("\nFQI Policy:")
evaluator_fqi.print_policy()
evaluator_fqi.compare_final_rewards()
evaluator_fqi.print_optimal_path(max_steps=30)

FQI: 100%|██████████| 1000/1000 [00:00<00:00, 4807.08it/s, theta_norm=47.6510]



FQI Policy:

========== LEARNED POLICY (FQI) ==========
  State 0: π(a=0|s=0) = 0.00  π(a=1|s=0) = 0.00  π(a=2|s=0) = 0.00  π(a=3|s=0) = 1.00  --> best action: 3
  State 1: π(a=0|s=1) = 0.00  π(a=1|s=1) = 1.00  π(a=2|s=1) = 0.00  π(a=3|s=1) = 0.00  --> best action: 1
  State 2: π(a=0|s=2) = 0.00  π(a=1|s=2) = 1.00  π(a=2|s=2) = 0.00  π(a=3|s=2) = 0.00  --> best action: 1
  State 3: π(a=0|s=3) = 0.00  π(a=1|s=3) = 0.00  π(a=2|s=3) = 0.00  π(a=3|s=3) = 1.00  --> best action: 3
  State 4: π(a=0|s=4) = 0.00  π(a=1|s=4) = 0.00  π(a=2|s=4) = 0.00  π(a=3|s=4) = 1.00  --> best action: 3
  State 5: π(a=0|s=5) = 0.00  π(a=1|s=5) = 1.00  π(a=2|s=5) = 0.00  π(a=3|s=5) = 0.00  --> best action: 1
  State 6: π(a=0|s=6) = 0.00  π(a=1|s=6) = 0.00  π(a=2|s=6) = 0.00  π(a=3|s=6) = 1.00  --> best action: 3
  State 7: π(a=0|s=7) = 0.00  π(a=1|s=7) = 0.00  π(a=2|s=7) = 0.00  π(a=3|s=7) = 1.00  --> best action: 3
  State 8: π(a=0|s=8) = 1.00  π(a=1|s=8) = 0.00  π(a=2|s=8) = 0.00  π(a=3|s=8) = 0.00  --> best

## 3 grid wall

In [19]:
states = torch.arange(9)
actions = torch.arange(4)
N = len(states) # number of states
A = len(actions) # number of actions
gamma = 0.9

x_0 = 0 # fixed initial state

goal = 8   # absorbing terminal state
pit = 5    # absorbing terminal state

def phi(x, a):
    # Convert to scalar if tensor
    if isinstance(x, torch.Tensor): x = x.item()
    if isinstance(a, torch.Tensor): a = a.item()

    vec = torch.zeros(N * A, dtype=torch.float64)
    vec[x * A + a] = 1.0
    return vec

step_cost = -0.1
goal_reward = 1.0
pit_reward  = -5.0

omega = torch.full((N * A,), step_cost, dtype=torch.float64)

# terminals: override step cost
omega[goal * A : goal * A + A] = goal_reward
omega[pit  * A : pit  * A + A] = pit_reward

# Helper to convert index <-> (row, col)
def to_rc(s):
    if isinstance(s, torch.Tensor): s = s.item()
    return divmod(s, 3)

def to_s(r, c): 
    return r*3 + c

wall = 4

def next_state(s, a):
    if isinstance(s, torch.Tensor): s = s.item()
    if isinstance(a, torch.Tensor): a = a.item()

    # absorbing terminals
    if s == goal or s == pit:
        return s

    r, c = to_rc(s)

    if a == 0:      # Up
        r2, c2 = max(0, r-1), c
    elif a == 1:    # Down
        r2, c2 = min(2, r+1), c
    elif a == 2:    # Left
        r2, c2 = r, max(0, c-1)
    elif a == 3:    # Right
        r2, c2 = r, min(2, c+1)

    sp = to_s(r2, c2)

    # wall blocks transition
    if sp == wall:
        return s

    return sp

def psi(xp):
    if isinstance(xp, torch.Tensor): xp = xp.item()
    
    v = torch.zeros(N * A, dtype=torch.float64)
    for x in states:
        for a in actions:
            if next_state(x, a) == xp:
                idx = x.item() * A + a.item()
                v[idx] = 1.0
    return v

# Initialize the solver with explicit terminal states
mdp = PolicySolver(
    states=states, 
    actions=actions, 
    phi=phi, 
    omega=omega, 
    gamma=gamma, 
    x0=x_0, 
    psi=psi,
    terminal_states=[goal, pit] 
)

In **terminal-aware dataset collection**, you must make sure the dataset contains **at least one transition with each terminal ((s,a))** as the *current* pair (typically a self-loop $(s,a,r(s,a),s)$ for every action at the terminal), because in linear FQI with one-hot $\phi(s,a)$ the regression can only learn the coordinate $\theta_{(s,a)}$ if $(s,a)$ appears on the left-hand side; otherwise that coordinate is **unidentifiable** and ridge keeps it at $\approx 0$, so $Q_\theta(s_{\text{term}},a)\approx 0$ for terminal states, the bootstrap term $\gamma\max_{a'}Q_\theta(s',a')$ becomes wrong whenever a transition enters a terminal, and this error propagates backwards through Bellman targets, making actions leading to goal/pit look almost the same and potentially yielding a policy that walks into the pit (or fails to seek the goal).


In [22]:
collector = EnvDataCollector(mdp=mdp, env_name="3grid_wall", max_steps=50)
collector.collect_dataset_terminal_aware(n_steps=3000, extra_steps=3, save_path=str(DATASET_PATH), verbose=True)

✅ Terminal-aware dataset saved to: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/test_fqi.csv
   Total transitions: 3000


,episode,step,state,action,reward,next_state
0,0,0,0,0,-0.1,0
1,0,1,0,3,-0.1,1
2,0,2,1,2,-0.1,0
3,0,3,0,1,-0.1,3
4,0,4,3,1,-0.1,6
...,...,...,...,...,...,...
2995,123,2,2,2,-0.1,1
2996,123,3,1,1,-0.1,1
2997,123,4,1,1,-0.1,1
2998,123,5,1,1,-0.1,1


In [24]:
solver_fqi = FQISolver(
    mdp=mdp,
    csv_path=str(DATASET_PATH),
    device=device,
    seed=seed,
    ridge=1e-6,
    augment_terminal_transitions=False
)
evaluator_fqi = FQIEvaluator(solver_fqi)
pi_fqi = solver_fqi.run(
    K=100, 
    tau=0.99, 
    verbose=True
)
print("\nFQI Policy:")
evaluator_fqi.print_policy()
evaluator_fqi.compare_value_functions()
evaluator_fqi.compare_final_rewards()
evaluator_fqi.print_optimal_path(max_steps=30)

FQI: 100%|██████████| 100/100 [00:00<00:00, 1625.80it/s, theta_norm=8.9967]


FQI Policy:

========== LEARNED POLICY (FQI) ==========
  State 0: π(a=0|s=0) = 0.00  π(a=1|s=0) = 1.00  π(a=2|s=0) = 0.00  π(a=3|s=0) = 0.00  --> best action: 1
  State 1: π(a=0|s=1) = 0.00  π(a=1|s=1) = 0.00  π(a=2|s=1) = 1.00  π(a=3|s=1) = 0.00  --> best action: 2
  State 2: π(a=0|s=2) = 0.00  π(a=1|s=2) = 0.00  π(a=2|s=2) = 1.00  π(a=3|s=2) = 0.00  --> best action: 2
  State 3: π(a=0|s=3) = 0.00  π(a=1|s=3) = 1.00  π(a=2|s=3) = 0.00  π(a=3|s=3) = 0.00  --> best action: 1
  State 4: π(a=0|s=4) = 1.00  π(a=1|s=4) = 0.00  π(a=2|s=4) = 0.00  π(a=3|s=4) = 0.00  --> best action: 0
  State 5: π(a=0|s=5) = 0.00  π(a=1|s=5) = 1.00  π(a=2|s=5) = 0.00  π(a=3|s=5) = 0.00  --> best action: 1
  State 6: π(a=0|s=6) = 0.00  π(a=1|s=6) = 0.00  π(a=2|s=6) = 0.00  π(a=3|s=6) = 1.00  --> best action: 3
  State 7: π(a=0|s=7) = 0.00  π(a=1|s=7) = 0.00  π(a=2|s=7) = 0.00  π(a=3|s=7) = 1.00  --> best action: 3
  State 8: π(a=0|s=8) = 1.00  π(a=1|s=8) = 0.00  π(a=2|s=8) = 0.00  π(a=3|s=8) = 0.00  --> best